[![Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://www.kaggle.com/code/maryrobynbernabe/kulintang-1-video-segmentation-silence-interval)

# Video Segmentation Based on Audio Silences

## Load the dataset

In [2]:
import os
for dirname, _, filenames in os.walk('/kaggle/input/datasets/maryrobynbernabe/kulintang-dataset/Kulintang UP MOT (Set 3)'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

/kaggle/input/datasets/maryrobynbernabe/kulintang-dataset/Kulintang UP MOT (Set 3)/IMG_0084.MOV
/kaggle/input/datasets/maryrobynbernabe/kulintang-dataset/Kulintang UP MOT (Set 3)/IMG_0090.MOV
/kaggle/input/datasets/maryrobynbernabe/kulintang-dataset/Kulintang UP MOT (Set 3)/IMG_0088.MOV
/kaggle/input/datasets/maryrobynbernabe/kulintang-dataset/Kulintang UP MOT (Set 3)/IMG_0091.MOV
/kaggle/input/datasets/maryrobynbernabe/kulintang-dataset/Kulintang UP MOT (Set 3)/IMG_0097.MOV
/kaggle/input/datasets/maryrobynbernabe/kulintang-dataset/Kulintang UP MOT (Set 3)/IMG_0093.MOV
/kaggle/input/datasets/maryrobynbernabe/kulintang-dataset/Kulintang UP MOT (Set 3)/IMG_0100.MOV
/kaggle/input/datasets/maryrobynbernabe/kulintang-dataset/Kulintang UP MOT (Set 3)/IMG_0098.MOV
/kaggle/input/datasets/maryrobynbernabe/kulintang-dataset/Kulintang UP MOT (Set 3)/IMG_0101.MOV
/kaggle/input/datasets/maryrobynbernabe/kulintang-dataset/Kulintang UP MOT (Set 3)/IMG_0086.MOV
/kaggle/input/datasets/maryrobynbernabe/

## Required installations

In [ ]:
# Install necessary packages
!pip install -q moviepy librosa ffmpeg-python

## Set the required paths

In [5]:
# constants

# path where you have folders of the videos you need to segment
VIDEO_FOLDER_PATH = "/kaggle/input/datasets/maryrobynbernabe/kulintang-dataset"
SEGMENTS_EXTRACTION_PATH = "VideoSegments"

## Required imports

In [7]:
import os
import moviepy.editor as mp
import librosa
import numpy as np
import ffmpeg
from tqdm.notebook import tqdm
from collections import defaultdict
from moviepy.video.io.VideoFileClip import VideoFileClip

/usr/local/lib/python3.12/dist-packages/moviepy/config_defaults.py:47: SyntaxWarning: invalid escape sequence '\P'
  IMAGEMAGICK_BINARY = r"C:\Program Files\ImageMagick-6.8.8-Q16\magick.exe"
/usr/local/lib/python3.12/dist-packages/moviepy/video/io/ffmpeg_reader.py:294: SyntaxWarning: invalid escape sequence '\d'
  lines_video = [l for l in lines if ' Video: ' in l and re.search('\d+x\d+', l)]
/usr/local/lib/python3.12/dist-packages/moviepy/video/io/ffmpeg_reader.py:367: SyntaxWarning: invalid escape sequence '\d'
  rotation_lines = [l for l in lines if 'rotate          :' in l and re.search('\d+$', l)]
/usr/local/lib/python3.12/dist-packages/moviepy/video/io/ffmpeg_reader.py:370: SyntaxWarning: invalid escape sequence '\d'
  match = re.search('\d+$', rotation_line)
error: XDG_RUNTIME_DIR not set in the environment.
ALSA lib confmisc.c:855:(parse_card) cannot find card '0'
ALSA lib conf.c:5178:(_snd_config_evaluate) function snd_func_card_inum returned error: No such file or directory
A

ModuleNotFoundError: No module named 'ffmpeg'

## Extract the Segments

### Required functions

In [13]:
from collections import defaultdict
# infomration list of extracted data file
meta_data = defaultdict(list)

# Function to extract folder
def extract_folder_path(path: str) -> str:
    """
    Extracts the folder name from a given path.
    :param path: str
        The path from which the folder name will be extracted.

    :return: str
    """

    return path.split("/")[-1]

def extrat_video_name(video_name: str) -> str:
    """
    Extracts the video name from a given video file name.
    :param video_name: str
        The video file name from which the video name will be extracted.

    :return: str
    """
    return video_name.split(".")[0]


def extract_silent_timestamps(input_video_path: str, audio_output_path: str, top_db: int = 20 )->list:
    """
    Extracts non-silent intervals from the audio of a video file.

    This function processes a video to extract its audio track using the `moviepy` library, then analyzes the audio to find intervals of sound using the `librosa` library. Silent sections are determined based on a decibel threshold, and only non-silent timestamps are returned.

    :param input_video_path: str
        The file path to the input video from which audio will be extracted.

    :param audio_output_path: str
        The file path where the extracted audio will be saved temporarily.

    :param top_db: int, optional
        The threshold in decibels for considering sections as non-silent. Defaults to 20. A lower value means more sensitive detection, identifying quieter sounds as non-silent.

    :return: List[Tuple[float, float]]
        A list of tuples, each representing the start and end times (in seconds) of non-silent intervals detected in the audio.

    Notes:
    - This method extracts and temporarily saves the audio from the video file, which is subsequently analyzed. Ensure that there is enough storage space for this temporary file.
    - The function removes any transient silent intervals that fall under the specified top_db threshold, which helps in focusing on meaningful non-silent parts of the audio.
    - Requires `moviepy` and `librosa` to be installed in the Python environment.
    """
    # Step 1: Extract audio from video using moviepy
    clip = mp.VideoFileClip(input_video_path)
    audio = clip.audio
    audio.write_audiofile(audio_output_path)

    # Step 2: Load audio and detect non-silent intervals using librosa
    y, sr = librosa.load(audio_output_path, sr=None)
    non_silent_intervals = librosa.effects.split(y, top_db=top_db)

    # Convert non-silent intervals to seconds
    non_silent_intervals_sec = [(float(start / sr), float(end / sr)) for start, end in non_silent_intervals]

    return non_silent_intervals_sec


def split_video(input_file: str, output_prefix: str, output_path: str, intervals: list, metainfo: tuple, metadata: dict = meta_data) -> dict:
    """
    Splits a video into multiple segments based on specified time intervals.

    :param input_file: The path to the input video file.
    :param output_prefix: The prefix for the output files.
    :param intervals: A list of tuples, each containing (start_time, end_time) in seconds.
    :param metainfo: list of metadata items
    :param metadata: metadata dictionary
    """
    with VideoFileClip(input_file) as video:
        for index, (start, end) in enumerate(intervals):
            # Calculate the duration for the current segment
            duration = end - start

            # Subclip the video
            subclip = video.subclip(start, end)

            # Construct the output filename
            output_file = f"{output_prefix}_part{index+1}.mov"

            # Set the output path
            set_output_path = os.path.join(output_path, output_file)
            print(metadata, type(metadata))

            # set the metadata
            metadata["video_name"].append(metainfo[0])
            metadata["video_root_folder"].append(metainfo[1])
            metadata["audio_name"].append(metainfo[2])
            metadata["audio_root_folder"].append(metainfo[3])
            metadata["video_segment_name"].append(output_file)
            metadata["video_segment_path"].append(set_output_path)
            metadata["label"].append("<Label: Note Number>")

            # Write the subclip to the file
            subclip.write_videofile(set_output_path, codec='libx264')

### Extract segments from each video

In [ ]:
from tqdm.notebook import tqdm
for root, dir, files in os.walk(VIDEO_FOLDER_PATH):
    for file_ in tqdm(files):
        # video root folder
        video_root_folder = os.path.join(root, file_)
        # extract the root directory where you have videos
        root_folder = extract_folder_path(root)
        # segment output path
        segment_folder = os.path.join(SEGMENTS_EXTRACTION_PATH, root_folder)
        # create path if not exists
        if not os.path.exists(segment_folder):
            os.makedirs(segment_folder)
            print("Folder created: {}".format(segment_folder))
        # extract video name
        video_name = extrat_video_name(file_)
        # create folders for each of the video names
        video_folder_path = os.path.join(segment_folder, video_name)
        # create the folder
        if not os.path.exists(video_folder_path):
            os.mkdir(video_folder_path)
            print("Folder Created: {}".format(video_folder_path))
        # save the audio and extract time feature
        audio_file_name = video_name + ".wav"
        # audio file path
        audio_file_path = os.path.join(video_folder_path, audio_file_name)
        # save the audio and extract the time stamp
        try:
          extracted_time_stamps = extract_silent_timestamps(video_root_folder, audio_file_path)
          # save the video segments
          split_video(
              video_root_folder,
              video_name,
              video_folder_path,
              extracted_time_stamps,
              (file_, video_root_folder, audio_file_name, audio_file_path),
          )
        except:
          print(f"{video_name} could not be processed")
          continue

0it [00:00, ?it/s]

  0%|          | 0/20 [00:00<?, ?it/s]

Folder created: VideoSegments/Kulintang UP MOT (Set 3)
Folder Created: VideoSegments/Kulintang UP MOT (Set 3)/IMG_0084
MoviePy - Writing audio in VideoSegments/Kulintang UP MOT (Set 3)/IMG_0084/IMG_0084.wav



chunk:  81%|████████  | 289/357 [00:00<00:00, 797.20it/s, now=None]
                                                                   

MoviePy - Done.
IMG_0084 could not be processed
Folder Created: VideoSegments/Kulintang UP MOT (Set 3)/IMG_0090
MoviePy - Writing audio in VideoSegments/Kulintang UP MOT (Set 3)/IMG_0090/IMG_0090.wav


...

MoviePy - Done.
IMG_0076 could not be processed
Folder Created: VideoSegments/Kulintang UP MOT (Set 2)/IMG_0074
MoviePy - Writing audio in VideoSegments/Kulintang UP MOT (Set 2)/IMG_0074/IMG_0074.wav



chunk:  83%|████████▎ | 318/382 [00:00<00:00, 926.66it/s, now=None]
                                                                   

MoviePy - Done.
IMG_0074 could not be processed
Folder Created: VideoSegments/Kulintang UP MOT (Set 2)/IMG_0069
MoviePy - Writing audio in VideoSegments/Kulintang UP MOT (Set 2)/IMG_0069/IMG_0069.wav



chunk:  79%|███████▉  | 286/362 [00:00<00:00, 844.17it/s, now=None]
                                                                   

MoviePy - Done.
IMG_0069 could not be processed
Folder Created: VideoSegments/Kulintang UP MOT (Set 2)/IMG_0073
MoviePy - Writing audio in VideoSegments/Kulintang UP MOT (Set 2)/IMG_0073/IMG_0073.wav


### Create the MetaData Dataframe

In [ ]:
import pandas as pd


# create the dataframe
metadata_df = pd.DataFrame.from_dict(meta_data)
metadata_df.head()

### Save the DataFrame

In [ ]:
save_path = "Kulingtang_metadata.csv"

# save the dataframe
metadata_df.to_csv(save_path, index = False)